# m01 lab · Bioelectric sources: transmembrane current, dipoles and double layers

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You build the sources of every electrogram from the ground up: a point current source, a source–sink pair and its
limit, the current dipole, a double layer and its solid angle, and finally a propagating wavefront, which turns out to
be a moving line of dipoles. The last part reproduces the unipolar electrogram of m03 from these pieces.

| Part | What you do | Equation |
| --- | --- | --- |
| A | A point source: Laplace's equation and current conservation | eq-m01-point |
| B | From a source–sink pair to a dipole | eq-m01-dipole |
| C | A double layer and its solid angle | eq-m01-double-layer |
| D | Membrane current across a wavefront | eq-m01-core |
| E | The wavefront as a line of dipoles | eq-m01-front-dipole |
| F | Why only wavefronts make potentials | eq-m01-core, eq-m01-double-layer |

Relative units throughout: conductivity $\sigma = 1$, $\beta = 1$, $\Delta V = 1$, distances in mm, as in m03. The
infinite homogeneous conductor is the standard starting model (Plonsey and Barr 2007). Each part ends with
**checks**. No downloads; an Octave copy of parts A–E is in `labs/octave/m01_lab.m`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import sources as S

## Part A · A point source

A point current source $I$ in an infinite conductor of conductivity $\sigma$ gives $\phi = I/(4\pi\sigma r)$
(eq-m01-point). Check it two ways: the finite-difference Laplacian vanishes away from the source, and the current
$\sigma E$ leaving a sphere around the source is $I$ whatever the sphere's radius.

In [ ]:
h = 1e-3
def lap(f, p):
    p = np.asarray(p, float)
    e = np.eye(3) * h
    return sum((f(p + e[i]) - 2 * f(p) + f(p - e[i])) / h**2 for i in range(3))

phi = lambda p: S.point_potential(1.0, 1.0, p)
pts = [(1.0, 0.0, 0.0), (0.3, -2.0, 1.5), (5.0, 5.0, 5.0)]
laps = [lap(phi, p) / abs(phi(p)) for p in pts]
print("Laplacian / potential at three points:", np.round(laps, 6))
fluxes = [S.flux_through_sphere(1.0, 1.0, R) for R in (0.5, 2.0, 10.0)]
print("current through spheres of radius 0.5, 2, 10:", np.round(fluxes, 5))

r = np.logspace(-1, 2, 100)
plt.loglog(r, S.point_potential(1.0, 1.0, np.stack([r, 0 * r, 0 * r], -1)))
plt.xlabel("distance (mm)"); plt.ylabel("potential (relative)"); plt.title("point source: 1/r"); plt.show()

In [ ]:
# Checks
assert max(abs(v) for v in laps) < 1e-4
assert all(abs(f - 1) < 1e-3 for f in fluxes)

## Part B · From a source–sink pair to a dipole

Put a source $+I$ and a sink $-I$ a distance $d$ apart. As $d \to 0$ with $p = Id$ fixed, the pair becomes a current
dipole, $\phi = \mathbf p\cdot(\mathbf r - \mathbf r_0)/(4\pi\sigma\lvert\mathbf r - \mathbf r_0\rvert^3)$
(eq-m01-dipole). Measure how fast the pair's potential falls: like a monopole close to one pole, like a dipole far
away.

In [ ]:
rho = np.logspace(-3, 3, 200)
obs = np.stack([0 * rho, 0 * rho, 0.5 + rho], -1)            # along the axis, measured from the source
slope = S.falloff_exponent(rho, S.pair_potential(1.0, 1.0, 1.0, obs))
plt.semilogx(rho, slope)
plt.xlabel("distance from the source pole (mm), pair separation 1 mm"); plt.ylabel("slope of log|phi|")
plt.show()
print(f"slope close to the pole {slope[0]:.3f}, far away {slope[-1]:.3f}")

pts = np.array([[0.3, 0.2, 5.0], [1.0, -2.0, 0.5], [4.0, 1.0, -3.0]])
errs = {}
for d in (0.5, 0.1, 0.02):
    approx = S.pair_potential(1.0 / d, d, 1.0, pts)
    exact = S.dipole_potential((0, 0, 1.0), 1.0, pts)
    errs[d] = np.max(np.abs(approx / exact - 1))
    print(f"separation {d:5.2f} mm: largest relative difference from the dipole {errs[d]:.2e}")

In [ ]:
# Checks
assert abs(slope[0] + 1) < 0.01 and abs(slope[-1] + 2) < 0.01
assert errs[0.02] < errs[0.1] < errs[0.5] and errs[0.02] < 1e-3

The error falls as the square of the separation: the next term in the expansion is a quadrupole.

## Part C · A double layer and its solid angle

Spread dipoles uniformly over a surface, with dipole moment $\tau$ per unit area. Each patch contributes
$\tau\,dA\cos\theta/(4\pi\sigma r^2)$, and $dA\cos\theta/r^2$ is the solid angle it subtends, so a uniform double layer
gives $\phi = \tau\,\Omega/(4\pi\sigma)$ (eq-m01-double-layer; the basis of solid-angle theory, Holland and Arnsdorf
1977). Check it on a disk of radius 2 mm, and watch the potential jump by $\tau/\sigma$ as you cross the layer.

In [ ]:
R = 2.0
cases = [(0.0, 1.0), (1.5, 0.5), (3.0, -2.0)]
rows = []
for x0, z0 in cases:
    om = S.disk_solid_angle(R, x0, z0)
    rows.append((x0, z0, om, S.double_layer_potential(1.0, 1.0, om), S.disk_dipole_sum(R, 1.0, 1.0, x0, z0)))
    print(f"point ({x0}, {z0}): solid angle {om:.4f} sr, tau Omega / 4 pi sigma {rows[-1][3]:.5f}, sum of dipoles {rows[-1][4]:.5f}")

z = np.concatenate([np.linspace(-4, -1e-3, 200), np.linspace(1e-3, 4, 200)])
plt.plot(z, S.double_layer_potential(1.0, 1.0, S.disk_solid_angle_axis(R, z)))
plt.xlabel("height on the disk axis (mm)"); plt.ylabel("potential (relative)"); plt.show()
jump = float(S.double_layer_potential(1.0, 1.0, S.disk_solid_angle_axis(R, 1e-6) - S.disk_solid_angle_axis(R, -1e-6)))
print(f"jump across the layer: {jump:.4f} (tau / sigma = 1)")

In [ ]:
# Checks
for x0, z0, om, a, b in rows:
    assert abs(a - b) < 1e-4 * max(abs(a), 1e-3)
assert abs(rows[0][2] - float(S.disk_solid_angle_axis(R, 1.0))) < 1e-8
assert abs(jump - 1.0) < 1e-5

## Part D · Membrane current across a wavefront

Along a thin strand, intracellular current flows down the voltage gradient, $I_i = -\beta\,\partial V_m/\partial x$, and
whatever it loses leaves through the membrane: $i_m = -\partial I_i/\partial x = \beta\,\partial^2 V_m/\partial x^2$
(eq-m01-core, the same result m03 uses). For the arctangent front of m03, compute $V_m$, $I_i$ and $i_m$, and check
that the membrane current sums to zero and that its first moment, the front's dipole moment, is $\beta\,\Delta V$ for
every width.

In [ ]:
s = np.linspace(-20000, 20000, 2000001)
moments = {}
for w in (0.1, 0.3, 1.0, 3.0):
    im = S.front_im(s, w)
    moments[w] = (S.net_current(s, im), S.dipole_moment(s, im))
    print(f"width {w:3.1f} mm: net membrane current {moments[w][0]:+.1e}, dipole moment {moments[w][1]:.4f}")

ss = np.linspace(-5, 5, 1001)
fig, ax = plt.subplots(1, 3, figsize=(11, 3))
ax[0].plot(ss, S.front_vm(ss, 0.3)); ax[0].set_title("V_m / dV")
ax[1].plot(ss, S.axial_current(ss, 0.3)); ax[1].set_title("intracellular current")
ax[2].plot(ss, S.front_im(ss, 0.3)); ax[2].set_title("membrane current (outward +)")
for a in ax: a.set_xlabel("s (mm), front at 0, resting tissue at s > 0")
plt.tight_layout(); plt.show()

In [ ]:
# Checks
for w, (net, p) in moments.items():
    assert abs(net) < 1e-9
    assert abs(p - 1.0) < 2e-3                         # beta dV, whatever the width
fine = np.linspace(-5, 5, 20001)                    # i_m = -dI_i/dx, by finite differences on a fine grid
i_num = -np.gradient(S.axial_current(fine, 0.3), fine)
assert np.max(np.abs(i_num - S.front_im(fine, 0.3))) < 1e-3 * np.max(np.abs(S.front_im(fine, 0.3)))

Ahead of the front the membrane current is outward (a source); behind it, inward (a sink). The pair is a dipole
pointing toward resting tissue, and its strength does not depend on how steep the front is.

## Part E · The wavefront as a line of dipoles

Seen edge-on, the front is a line along $y$; each strip of width $dx$ is a line source with potential
$-\ln r/(2\pi\sigma)$ per unit current. Sum the strips at an electrode a height $h$ above the tissue and compare with
the m03 closed form $U = A s/(s^2 + (w + h)^2)$, $A = \beta\Delta V/(2\pi\sigma)$. Then compare with a line of dipoles,
$\phi = p\,s/(2\pi\sigma(s^2 + h^2))$ (eq-m01-front-dipole).

In [ ]:
ss = np.linspace(-8, 8, 33)
w, hh = 0.3, 1.0
quad = S.front_potential_quadrature(ss, w, hh)
closed = S.front_potential(ss, w, hh)
line_wh = S.line_dipole_potential(1.0, 1.0, ss, w + hh)
line_h = S.line_dipole_potential(1.0, 1.0, ss, hh)
err_quad = np.max(np.abs(quad - closed)) / np.max(np.abs(closed))
err_line = np.max(np.abs(line_wh - closed)) / np.max(np.abs(closed))
print(f"sum of strips vs m03 closed form: {err_quad:.1e}; line of dipoles at height w + h vs closed form: {err_line:.1e}")
plt.plot(ss, closed, "k-", label="m03 closed form")
plt.plot(ss, quad, "o", ms=4, label="sum of membrane-current strips")
plt.plot(ss, line_h, "--", label="thin front (line of dipoles) at height h")
plt.xlabel("s (mm)"); plt.ylabel("potential (relative)"); plt.legend(fontsize=8); plt.show()
far = np.array([50.0, 200.0])
far_ratio = S.front_potential(far, w, hh) / S.line_dipole_potential(1.0, 1.0, far, 0.0)
print("far away, front over a bare line dipole:", np.round(far_ratio, 4))

In [ ]:
# Checks
assert err_quad < 1e-6 and err_line < 1e-12
assert np.all(np.abs(far_ratio - 1) < 1e-3)

A front of width $w$ at height $h$ produces exactly the field of an infinitely thin front, a line of dipoles, at
height $w + h$. Widening the front and lifting the electrode do the same thing.

## Part F · Why only wavefronts make potentials

Where $V_m$ is uniform, fully at rest or fully activated, $\partial^2 V_m/\partial x^2 = 0$ and no membrane current
flows, so no potential is produced. In solid-angle terms, a uniformly activated region is bounded by a closed double
layer, and a closed surface seen from outside subtends zero solid angle. Compare a strand with a front on it to one
that is entirely activated.

In [ ]:
x = np.linspace(-50, 50, 20001)
plateau = np.full_like(x, 1.0)                        # every cell activated, V_m uniform
im_plateau = np.gradient(np.gradient(plateau, x), x)
print("largest membrane current, uniform strand:", float(np.max(np.abs(im_plateau))))
print("largest membrane current, strand with a front:", float(np.max(np.abs(S.front_im(x, 0.3)))))
two = S.disk_solid_angle(2.0, 5.0, 0.4) + S.disk_solid_angle(2.0, 5.0, -0.4)
print(f"two opposite disks (the faces of a thin activated slab) seen from the side: {two:.2e} sr")

In [ ]:
# Checks
assert np.max(np.abs(im_plateau)) == 0.0
assert abs(two) < 1e-9

This is why the surface ECG is nearly flat while the ventricles are uniformly depolarized, and why electrograms
come from wavefronts, not from activated or resting tissue as such.

## Where to go next

- In part B, put the observation point on the dipole's perpendicular bisector. What is the potential, and why?
- In part C, move the observation point from the axis out past the rim of the disk at a fixed small height. Where
  does the potential change fastest?
- In part E, keep $w + h$ fixed and trade width for height. Does the electrogram change?
- m02 replaces the infinite conductor with blood, tissue and boundaries; m03 turns these potentials into unipolar,
  bipolar and omnipolar electrograms.